# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [5]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import certifi

In [6]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

API key looks good so far


In [7]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers, verify=False)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [8]:
ed = Website("https://edwarddonner.com")
ed.links

C:\Users\aalperen.arda\AppData\Local\anaconda3\envs\llms\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'edwarddonner.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/',
 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/',
 'https://edwarddonner.com/2025/04/21/the-complete-agentic-ai-engineering-course/',
 'https://edwarddonner.com/2025/04/21/the-

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [9]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}
"""

In [10]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}



In [11]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [12]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://edwarddonner.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/
https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/
https://edwarddo

In [13]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [14]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website("https://huggingface.co")
huggingface.links

C:\Users\aalperen.arda\AppData\Local\anaconda3\envs\llms\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


['/',
 '/models',
 '/datasets',
 '/spaces',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/black-forest-labs/FLUX.1-Kontext-dev',
 '/THUDM/GLM-4.1V-9B-Thinking',
 '/google/gemma-3n-E4B-it',
 '/kyutai/tts-1.6b-en_fr',
 '/tencent/Hunyuan-A13B-Instruct',
 '/models',
 '/spaces/enzostvs/deepsite',
 '/spaces/black-forest-labs/FLUX.1-Kontext-Dev',
 '/spaces/ilcve21/Sparc3D',
 '/spaces/AIDC-AI/Ovis-U1-3B',
 '/spaces/tencent/Hunyuan3D-2.1',
 '/spaces',
 '/datasets/fka/awesome-chatgpt-prompts',
 '/datasets/HuggingFaceFW/fineweb-2',
 '/datasets/facebook/seamless-interaction',
 '/datasets/marcelbinz/Psych-101',
 '/datasets/black-forest-labs/kontext-bench',
 '/datasets',
 '/join',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/allenai',
 '/facebook',
 '/amazon',
 '/google',
 '/Intel',
 '/microsoft',
 '/grammarly',
 '/Writer',
 '/docs/tra

In [15]:
get_links("https://huggingface.co")

C:\Users\aalperen.arda\AppData\Local\anaconda3\envs\llms\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'documentation page', 'url': 'https://huggingface.co/docs'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [16]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [17]:
print(get_all_details("https://huggingface.co"))

C:\Users\aalperen.arda\AppData\Local\anaconda3\envs\llms\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\aalperen.arda\AppData\Local\anaconda3\envs\llms\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/about'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'community page', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}


C:\Users\aalperen.arda\AppData\Local\anaconda3\envs\llms\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\aalperen.arda\AppData\Local\anaconda3\envs\llms\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'apply.workable.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\aalperen.arda\AppData\Local\anaconda3\envs\llms\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-us

Landing page:
Webpage Title:
Hugging Face – The AI community building the future.
Webpage Contents:
Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 1M+ models
Trending on
this week
Models
black-forest-labs/FLUX.1-Kontext-dev
Updated
9 days ago
•
163k
•
1.38k
THUDM/GLM-4.1V-9B-Thinking
Updated
4 days ago
•
8.58k
•
234
google/gemma-3n-E4B-it
Updated
4 days ago
•
212k
•
489
kyutai/tts-1.6b-en_fr
Updated
4 days ago
•
9.29k
•
204
tencent/Hunyuan-A13B-Instruct
Updated
5 days ago
•
20.8k
•
733
Browse 1M+ models
Spaces
Running
9.36k
9.36k
DeepSite v2
🐳
Generate any application with DeepSeek
Running
on
Zero
MCP
620
620
FLUX.1 Kontext
⚡
Kontext image editing on FLUX[dev]
Running
1.07k
1.07k
Sparc3D
🏃
Next-Gen High-Resolution 3D Model Generation
Running
on
Zero
138
138
Ovis U1 3B
🎨
Demo for multim

In [32]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown and in Turkish.\
Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."


In [33]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:10_000] # Truncate if more than 5,000 characters
    return user_prompt

In [40]:
get_brochure_user_prompt("Arda Gıda", "https://samiardagida.com")

Found links: {'links': [{'type': 'about page', 'url': 'https://www.samiardagida.com/'}, {'type': 'contact page', 'url': 'https://www.samiardagida.com/bize-ulasin/'}, {'type': 'careers page', 'url': 'https://www.samiardagida.com/my-account/'}]}


'You are looking at a company called: Arda Gıda\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nToptan Gıda Ürünleri Online Satış Mağazası | Sami Arda Gida\nWebpage Contents:\nİçeriğe atla\n+90 ( 850 )\n888 22 39\nWhatsapp\n0543 477 99 59\nSipariş Takibi\nBize Ulaşın\nWhatsapp\n0543 477 99 59 (Tıklayınız)\nAra:\n0,00\n₺\nSepetinizde ürün bulunmuyor.\nMağazaya geri dön\nSepet\nSepetinizde ürün bulunmuyor.\nMağazaya geri dön\n1000 TL\nve üzeri alışverişlerde kargo bedava!\nŞARKÜTERİ\nTEMEL GIDA\nTATLI & PASTA\nİÇECEK\nCATERING\nKONSERVELER\nBAHARAT\nEDİRNE YÖRESEL\nHAFTANIN ÜRÜNLERİ\n-10%\nAz Tuzlu\xa0Gemlik\xa0Zeytin 10 kg Teneke [321-360 kalibre]\n1.980,00\n₺\nOrijinal fiyat: 1.980,00₺.\n1.780,00\n₺\nŞu andaki fiyat: 1.780,00₺.\nSepete Ekle\n-13%\nÖzden Piliç Sosis 800 gr\n149,90\n₺\nOrijinal fiyat: 149,90₺.\n129,90\n₺\nŞu andaki fiyat: 129,90₺.\nSepete Ekle\n

In [41]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [42]:
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

create_brochure("Arda Gıda", "https://samiardagida.com")

Found links: {'links': [{'type': 'company page', 'url': 'https://www.samiardagida.com/'}, {'type': 'contact page', 'url': 'https://www.samiardagida.com/bize-ulasin/'}, {'type': 'products page', 'url': 'https://www.samiardagida.com/urunlerimiz/'}, {'type': 'careers page', 'url': 'https://www.samiardagida.com/'}, {'type': 'order tracking page', 'url': 'https://www.samiardagida.com/siparis-takibi/'}]}


# Arda Gıda Broşürü

## Şirket Tanıtımı
**Sami Arda Gıda**, toptan gıda ürünlerinin online satışını gerçekleştiren bir şirkettir. Geniş ürün yelpazesiyle müşterilerine çeşitli seçenekler sunmaktadır. Şarküteri, temel gıda, tatlı& pasta, içecekler, catering, konserveler ve baharatlar gibi kategorilerde kaliteli ürünler sunarak gıda sektöründe kendine sağlam bir yer edinmiştir.

## Ürünlerimiz

- **Şarküteri**: Geleneksel ve lezzetli et ürünleri
- **Temel Gıda**: Bakliyatlar, unlar ve diğer temel gıda malzemeleri
- **Tatlı & Pasta**: Hazır tatlılar ve pasta malzemeleri
- **İçecek**: Sıcak ve soğuk içecek çeşitleri
- **Catering**: Özel davetler için catering ürünleri
- **Konserveler**: Sebze, meyve ve balık konserveleri
- **Baharata**: Türkiye'nin çeşitli bölgelerinden gelen seçkin baharatlar

Müşterilerimiz, 1000 TL ve üzeri alışverişlerde kargo bedava avantajının da keyfini çıkarabilirler.

## Müşteri İlişkilere
Müşteri memnuniyeti bizim için son derece önemlidir. Müşterilerimiz istedikleri ürünleri almak için +90 (850) 888 22 39 veya WhatsApp üzerinden bizlerle iletişime geçebilir.

## İş Kültürü
Arda Gıda, çalışma ortamında saygı, takım çalışması ve yenilikçiliği teşvik eden bir kültüre sahiptir. Çalışanlarımızı sürekli gelişim için desteklemekte ve onların kariyer hedeflerine ulaşmalarında yardımcı olmaktayız. Hızla büyüyen dinamik bir ekip içinde yer alma fırsatı sunuyoruz.

## Kariyer Fırsatları
Arda Gıda, kariyerini geliştirmek isteyen yeni yeteneklere kapılarını açmaktadır. Açık pozisyonlar hakkında bilgi almak ve başvurmak için lütfen web sitemizi ziyaret edin.

## İletişim
Bize ulaşmak için:
- Telefon: +90 (850) 888 22 39
- WhatsApp: 0543 477 99 59

Arda Gıda ile lezzet dolu bir dünyaya adım atmaya hazır mısınız? Şimdi alışveriş yapın ve kaliteyi deneyimleyin!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [44]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

# You can use Win + / for multi selection

In [45]:
stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'company page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}


# Hugging Face Broşürü

## Şirket Hakkında
Hugging Face, yapay zeka (AI) topluluğunun geleceği inşa etmek için bir araya geldiği bir platformdur. 2016 yılında kurulan şirket, makine öğrenimi, doğal dil işleme ve derin öğrenme alanında uzmanlaşmış olup, Paris, Fransa merkezlidir. Hugging Face, dünya çapında 50,000'den fazla organizasyona hizmet veren bir iş modeli sunmaktadır.

## Şirket Kültürü
Hugging Face, toplum tabanlı bir yaklaşımla, herkesin etkili bir şekilde AI ve makine öğrenimi uygulamaları geliştirmesi için bir ortam sağlamayı hedeflemektedir. Şirket, ortak çalışmalar ve açık kaynaklı araçlarla, makine öğrenimini demokratikleştirmekte ve topluluğun gelişimine katkıda bulunmaktadır. "Her bir katkı ile daha iyi bir makine öğrenimi sağlamak" misyonuyla hareket eden Hugging Face, dinamik ve çeşitliliğe açık bir iş kültürü oluşturmayı amaçlamaktadır.

## Ürünler ve Hizmetler
- **Modeller:** 1 milyondan fazla makine öğrenimi modeli barındıran platform; kullanıcılar, bu modeller üzerinde çalışmalarını sürdürebilir ve işbirliği yapabilir.
- **Veri Setleri:** 250,000’den fazla veri seti sunarak, kullanıcıların her türlü makine öğrenimi görevinde ihtiyaçlarını karşılamaktadır.
- **Spaces:** Kullanıcıların AI uygulamaları oluşturup çalıştırabileceği bir alan.
- **Enterprise Çözümleri:** Kurumsal düzeyde güvenlik ve destek sunarak, takımlara özel çözümler geliştirmektedir.

## Müşteriler
Hugging Face, Amazon, Google, Microsoft ve Intel gibi büyük teknoloji şirketleri dahil olmak üzere, çok çeşitli sektörlerdeki organizasyonlarla işbirliği yapmaktadır. Şirket, ayrıca daha küçük organizasyonlar ve bireysel geliştiricilerle de iş yaparak geniş çapta bir etki yaratmayı hedeflemektedir.

## Kariyer ve İş Olanakları
Hugging Face, sürekli büyüyen bir ekibe sahiptir. Çeşitli pozisyonlarda yetenekli bireylere kapı açan şirket, adaylarına yenilikçi bir çalışma ortamı sunmaktadır. Çalışanları, gelişim ve öğrenim fırsatları ile desteklenmektedir. Kariyer olanaklarına ulaşmak ve başvuruda bulunmak için [kariyer sayfasını](https://huggingface.co) ziyaret edebilirsiniz.

## İletişim
Daha fazla bilgi ve güncellemeler için lütfen [Hugging Face resmi web sitesini](https://huggingface.co) ziyaret edin veya sosyal medya üzerindeki hesaplarını takip edin.

---

Hugging Face, yapay zeka alanında yenilikçi çözümler sunduğu gibi, aynı zamanda bu alandaki topluluğu güçlendirmeye de odaklanmaktadır. Sizi, bu heyecan verici yolculuğa katılmaya davet ediyoruz!

In [46]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")
# Bunlar bende calismiyor cunku sirket agindayim.

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/about'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'company page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}, {'type': 'community discussion', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub', 'url': 'https://github.com/huggingface'}, {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'}, {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'}]}


# Hugging Face Broşürü

## Şirket Hakkında
**Hugging Face**, yapay zeka ve makine öğrenimi topluluğunu bir araya getirerek geleceği inşa eden bir platformdur. Milyonlarca model, veri seti ve uygulama ile kullanıcıların iş birliği yapmasına olanak tanır. Topluluğumuz, makine öğrenmeyi demokratikleştirme misyonuna sahiptir ve herkesin AI araçlarına erişimini sağlamaktadır.

## Şirket Kültürü
Hugging Face, açıklık ve iş birliğini teşvik eden bir kültüre sahiptir. Çalışanlarına, yaratıcı süreçlerde aktif rol alma fırsatı tanımaktadır. Açık kaynak topluluğu ile birlikte çalışmak ve bilgi paylaşımında bulunmak, Hugging Face’in DNA’sının bir parçasıdır. Sürekli öğrenme ve yenilik, çalışma ortamının temellerini oluşturur.

## Müşterilerimiz
Hugging Face, dünya çapında 50,000'den fazla organizasyona hizmet vermektedir, bunlar arasında:

- **Google**
- **Amazon**
- **Microsoft**
- **Meta**
- **Intel**

Bu büyük işletmeler, Hugging Face'in sağladığı güçlü AI ve makine öğrenimi çözümlerini kullanarak kendi projelerinde yenilik yapmaktadırlar.

## Kariyer ve İş İmkanları
Hugging Face, yetenekli bireyleri takıma katılmaya davet etmektedir. Sürekli büyüyen bu ekibe katılarak, dünya çapında bir etki yaratma fırsatını yakalayabilirsiniz. Şu anda açık pozisyonlarımız arasında:

- Yapay Zeka Araştırmacısı
- Yazılım Mühendisi
- Veri Bilimci

Kariyer olanakları ve staj programları hakkında daha fazla bilgi için [kariyer sayfamızı ziyaret edebilirsiniz](https://huggingface.co/careers).

## Ürünlerimiz
- **Modeller**: 1 milyondan fazla model keşfedin.
- **Veri Setleri**: 250.000'den fazla veri setine erişin.
- **Uygulamalar**: Farklı alanlarda AI uygulamaları oluşturun ve paylaşın.
- **Kurumsal Çözümler**: Takımlara özel güvenli platformlar sunarak AI geliştirme süreçlerini hızlandırın.

## Topluluk
Hugging Face, kullanıcıların bir araya geldiği ve bilgi alışverişinde bulunduğu bir topluluktur. Forumlar, bloglar ve makaleler aracılığıyla makine öğrenimi konusunda fikir alışverişinde bulunabilir, öğrenebilir ve gelişebilirsiniz.

**Hugging Face ile AI dünyasında bir adım önde olun!** 

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>